## Підбор параметра partitionBlock та filterBlock
Зафіксовані параметри під час підбору:
- `threshold` (базовий випадок в послідовному): 8192
- `parallelSortCutoff` (поріг, коли включається паралельне сортування): 0 - значить, що паралельного сортування не буде
- `parallelCutoff`: 131072

`partitionBlock` підбирався на параметрах 4096, 8192, 16384, 32768, 65536.

`filterBlock` підбирався на параметрах 1024, 2048, 4096, 8192, 16384, 32768.

Для вибору розмірів блоків застосовувався той самий критерій, що й для порогу паралельності. Найменше середнє геометричне відносного часу (1,007) мала комбінація з блоком розбиття 8192 і блоком фільтра 2048. Комбінації 8192, 1024 і 8192, 4096 відставали від неї менш ніж на 0,5%, що не перевищує розкиду (1.4%) між повторами.

### Обране значення partitionBlock: `8192`
### Обране значення filterBlock: `2048`

In [1]:
# Дані
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
a = pd.read_csv("C:/Users/admin/source/repos/Kruskal/results/fk_par_blocks.csv").assign(run="1: великі")
b = pd.read_csv("C:/Users/admin/source/repos/Kruskal/results/fk_par_blocks_small.csv").assign(run="2: малі")
df = pd.concat([a, b], ignore_index=True)

MIN_EDGES = 1_000_000
big = df[df["E"] >= MIN_EDGES]              # поріг впливає лише на графи, де ребер помітно більше за нього

In [5]:
graphs = (
    df[["series", "graph", "V", "E"]]
    .drop_duplicates()
    .sort_values(["series", "V", "E"])
    .reset_index(drop=True)
)

display(graphs)

,series,graph,V,E
0,density,B1,8000,16000
1,density,B2,8000,32000
2,density,B3,8000,64000
3,density,B4,8000,128000
4,density,B5,8000,256000
5,density,B6,8000,512000
6,density,B7,8000,1024000
7,density,B8,8000,2048000
8,density,B9,8000,4096000
9,density,B10,8000,8192000


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7505 entries, 0 to 7504
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   series           7505 non-null   object 
 1   graph            7505 non-null   object 
 2   family           7505 non-null   object 
 3   algorithm        7505 non-null   object 
 4   threads          7505 non-null   int64  
 5   threshold        7505 non-null   int64  
 6   partition_block  7505 non-null   int64  
 7   filter_block     7505 non-null   int64  
 8   par_cutoff       7505 non-null   int64  
 9   psort_cutoff     7505 non-null   int64  
 10  V                7505 non-null   int64  
 11  E                7505 non-null   int64  
 12  E_over_V         7505 non-null   float64
 13  seed             7505 non-null   int64  
 14  repeat           7505 non-null   int64  
 15  total_ms         7505 non-null   float64
 16  sort_ms          0 non-null      float64
 17  dsu_ms        

In [7]:
# Параметри досліду
PARAMS = ["threads", "threshold", "partition_block", "filter_block", "par_cutoff", "psort_cutoff"]

def experiment_params(df):
    rows = {}
    for alg, d in df.groupby("algorithm", sort=False):
        row = {}
        for c in PARAMS:
            vals = sorted(d[c].dropna().unique())
            row[c] = ", ".join(str(int(v)) for v in vals) if vals else "-"
        repeats = d.groupby(["graph"] + PARAMS, dropna=False)["repeat"].nunique()
        row["повторів"] = f"{repeats.min()}–{repeats.max()}" if repeats.min() != repeats.max() else str(repeats.min())
        rows[alg] = row
    graphs = df["graph"].unique()
    print(f"Графи ({len(graphs)}): {', '.join(graphs)}")
    return pd.DataFrame(rows).T

In [8]:
# 1. Зафіксовані параметри кожного прогону (функція experiment_params з попередньої клітинки)
for run, d in df.groupby("run"):
    print(f"\n=== Прогін {run}")
    display(experiment_params(d))


=== Прогін 1: великі
Графи (19): B1, B2, B3, B4, B5, B6, B7, B8, B9, B10, B11, G1, C1, C2, C3, C4, C5, C6, C7


,threads,threshold,partition_block,filter_block,par_cutoff,psort_cutoff,повторів
fk-par,12,8192,"16384, 32768, 65536","4096, 8192, 16384, 32768",131072,0,7–25



=== Прогін 2: малі
Графи (19): B1, B2, B3, B4, B5, B6, B7, B8, B9, B10, B11, G1, C1, C2, C3, C4, C5, C6, C7


,threads,threshold,partition_block,filter_block,par_cutoff,psort_cutoff,повторів
fk-par,12,8192,"4096, 8192, 16384","1024, 2048, 4096",131072,0,7–25


In [9]:
# Аналіз розмірів блоків: та сама клітинка, але для кожного прогону окремо
order = list(df["graph"].unique())   # порядок графів як у плані


def analyze_blocks(df):
    # Середній час fk-par для кожної комбінації блоків
    par = df[df["algorithm"] == "fk-par"].copy()
    par["partition_block"] = par["partition_block"].astype(int)
    par["filter_block"] = par["filter_block"].astype(int)
    t = par.groupby(["graph", "E", "partition_block", "filter_block"], as_index=False)["total_ms"].mean()

    t["rel"] = t["total_ms"] / t.groupby("graph")["total_ms"].transform("min")   # 1.00 = найкраща комбінація блоків на графі

    print("Середній час fk-par, мс:")
    display(t.pivot(index="graph", columns=["partition_block", "filter_block"], values="total_ms").reindex(order).round(1))

    big = t[t["E"] >= MIN_EDGES]
    big_order = [g for g in order if g in set(big["graph"])]
    print(f"Графи з E >= {MIN_EDGES:,}: {', '.join(big_order)}")

    rel_table = big.pivot(index="graph", columns=["partition_block", "filter_block"], values="rel").reindex(big_order)
    print("Відносний час (1.00 = найкраща комбінація блоків на цьому графі):")
    display(rel_table.round(3))

    summary = (big.groupby(["partition_block", "filter_block"])["rel"]
                  .agg(середнє_відставання=lambda r: np.exp(np.log(r).mean()),
                       найгірший_граф="max",
                       перемог=lambda r: int((r == 1.0).sum()))
                  .sort_values("середнє_відставання"))
    display(summary.round(3))

    best = summary.index[0]
    print(f"\nНайкращий розмір блоку: partition={best[0]}, filter={best[1]}")
    print(f"Він повільніший за найкращий на кожному графі в середньому на {100 * (summary.iloc[0, 0] - 1):.1f}%")

    noise = (par[par["E"] >= MIN_EDGES].groupby(["graph", "partition_block", "filter_block"])["total_ms"]
                .agg(lambda x: x.std() / x.mean()).median() * 100)
    print(f"Типовий розкид між повторами: {noise:.1f}%  (різниця, менша за нього - це шум)")


for run, d in df.groupby("run"):
    print(f"\n========== Прогін {run} ==========")
    analyze_blocks(d)


========== Прогін 1: великі ==========
Середній час fk-par, мс:


partition_block   16384                           32768                  \
filter_block      4096    8192    16384   32768   4096    8192    16384   
graph                                                                     
B1                  0.9     0.9     0.9     0.9     0.9     0.9     0.9   
B2                  1.1     1.1     1.2     1.1     1.0     1.0     1.0   
B3                  1.4     1.4     1.4     1.4     1.4     1.5     1.4   
B4                  2.3     2.2     2.3     2.3     2.3     2.2     2.3   
B5                  3.6     3.7     3.6     3.7     3.6     3.7     3.6   
B6                  5.5     5.5     5.5     5.5     5.4     5.4     5.4   
B7                  9.6     9.9    10.0    10.0    10.1    10.1    10.2   
B8                 18.4    18.2    18.5    18.8    18.9    19.0    18.5   
B9                 37.2    37.5    37.7    37.8    37.0    36.8    36.8   
B10                80.1    81.3    81.7    80.7    80.3    79.7    79.7   
B11               309.9   314.0   311.9   317.3   306.2   309.9   312.2   
G1                912.4   909.6   910.1   920.5   897.7   898.8   906.6   
C1                  1.8     1.7     1.8     1.7     1.7     1.8     1.7   
C2                  6.4     6.4     6.4     6.5     6.1     6.2     6.2   
C3                 18.3    18.1    18.5    19.0    18.2    17.7    18.4   
C4                 54.9    54.8    55.3    56.1    55.2    55.2    56.1   
C5                200.8   202.8   207.9   212.8   209.3   204.9   207.7   
C6                891.8   869.1   868.4   867.6   854.1   868.4   876.1   
C7               3591.1  3595.9  3622.7  3653.9  3601.2  3668.0  3616.5   

partition_block           65536                          
filter_block      32768   4096    8192    16384   32768  
graph                                                    
B1                  0.9     0.9     0.9     0.9     0.9  
B2                  1.0     1.0     1.0     1.0     1.0  
B3                  1.4     1.4     1.4     1.4     1.4  
B4                  2.2     2.3     2.3     2.3     2.3  
B5                  3.6     3.6     3.6     3.6     3.6  
B6                  5.5     5.8     5.8     5.8     5.8  
B7                 10.4    11.0    10.8    10.8    10.5  
B8                 18.8    19.5    19.4    19.5    19.5  
B9                 36.6    37.4    38.2    37.7    37.8  
B10                79.0    79.2    78.9    79.2    79.5  
B11               311.7   315.1   313.7   307.2   309.6  
G1                920.6   910.5   920.6   915.4   938.4  
C1                  1.8     1.8     1.8     1.8     1.8  
C2                  6.2     6.2     6.3     6.3     6.3  
C3                 18.8    18.5    18.8    18.7    18.9  
C4                 57.1    55.5    55.7    55.9    57.0  
C5                209.7   209.7   211.2   213.5   211.5  
C6                880.2   871.8   872.9   875.4   881.3  
C7               3638.1  3626.3  3636.4  3683.3  3700.9

Графи з E >= 1,000,000: B7, B8, B9, B10, B11, G1, C4, C5, C6, C7
Відносний час (1.00 = найкраща комбінація блоків на цьому графі):


partition_block  16384                       32768                       \
filter_block     4096   8192   16384  32768  4096   8192   16384  32768   
graph                                                                     
B7               1.000  1.025  1.034  1.035  1.050  1.051  1.061  1.076   
B8               1.009  1.000  1.014  1.033  1.035  1.041  1.016  1.030   
B9               1.017  1.026  1.030  1.033  1.010  1.004  1.006  1.000   
B10              1.016  1.030  1.035  1.023  1.018  1.011  1.010  1.001   
B11              1.012  1.026  1.019  1.036  1.000  1.012  1.020  1.018   
G1               1.016  1.013  1.014  1.025  1.000  1.001  1.010  1.025   
C4               1.001  1.000  1.008  1.023  1.007  1.007  1.023  1.042   
C5               1.000  1.010  1.035  1.060  1.042  1.020  1.034  1.044   
C6               1.044  1.018  1.017  1.016  1.000  1.017  1.026  1.030   
C7               1.000  1.001  1.009  1.017  1.003  1.021  1.007  1.013   

partition_block  65536                       
filter_block     4096   8192   16384  32768  
graph                                        
B7               1.146  1.125  1.122  1.095  
B8               1.069  1.064  1.069  1.069  
B9               1.022  1.043  1.030  1.033  
B10              1.004  1.000  1.004  1.007  
B11              1.029  1.025  1.003  1.011  
G1               1.014  1.025  1.020  1.045  
C4               1.013  1.016  1.020  1.040  
C5               1.044  1.051  1.063  1.053  
C6               1.021  1.022  1.025  1.032  
C7               1.010  1.013  1.026  1.031

середнє_відставання  найгірший_граф  перемог
partition_block filter_block                                              
16384           4096                        1.011           1.044        3
                8192                        1.015           1.030        2
32768           4096                        1.016           1.050        3
                8192                        1.018           1.051        0
                16384                       1.021           1.061        0
16384           16384                       1.021           1.035        0
32768           32768                       1.028           1.076        1
16384           32768                       1.030           1.060        0
65536           4096                        1.036           1.146        0
                16384                       1.038           1.122        0
                8192                        1.038           1.125        1
                32768                       1.041           1.095        0


Найкращий розмір блоку: partition=16384, filter=4096
Він повільніший за найкращий на кожному графі в середньому на 1.1%
Типовий розкид між повторами: 1.5%  (різниця, менша за нього - це шум)

========== Прогін 2: малі ==========
Середній час fk-par, мс:


partition_block   4096                    8192                    16384  \
filter_block       1024    2048    4096    1024    2048    4096    1024   
graph                                                                     
B1                  0.8     0.8     0.8     0.8     0.8     0.8     0.8   
B2                  1.0     1.0     1.0     1.0     1.0     1.0     1.0   
B3                  1.3     1.3     1.3     1.3     1.3     1.3     1.3   
B4                  2.3     2.3     2.2     2.2     2.2     2.2     2.2   
B5                  3.6     3.7     3.6     3.7     3.7     3.8     3.7   
B6                  5.4     5.4     5.3     5.7     5.7     5.7     5.5   
B7                  9.3     9.3     9.2     9.1     9.3     9.3     9.2   
B8                 17.5    17.7    17.6    17.5    17.4    17.5    17.4   
B9                 32.3    32.5    32.4    32.6    32.8    32.9    33.1   
B10                71.0    70.9    70.7    70.6    70.6    70.5    72.7   
B11               312.3   304.1   302.9   282.3   277.3   279.2   291.4   
G1                896.1   819.9   815.8   813.9   815.7   839.1   886.6   
C1                  1.9     1.9     1.7     1.7     1.7     1.8     1.7   
C2                  6.2     6.4     6.3     6.4     6.3     6.3     6.4   
C3                 18.3    18.2    18.5    18.8    18.4    18.5    18.2   
C4                 53.5    53.2    53.8    53.4    51.8    53.2    52.9   
C5                192.4   191.8   193.7   187.6   191.3   187.5   192.7   
C6                792.1   777.4   779.9   785.5   778.6   779.6   781.9   
C7               3344.5  3332.0  3324.3  3352.7  3330.0  3325.3  3335.0   

partition_block                  
filter_block       2048    4096  
graph                            
B1                  0.8     0.8  
B2                  1.0     1.0  
B3                  1.3     1.3  
B4                  2.2     2.2  
B5                  3.7     3.6  
B6                  5.4     5.5  
B7                  9.1     9.1  
B8                 17.5    17.4  
B9                 33.2    33.3  
B10                72.8    73.1  
B11               285.0   286.4  
G1                828.2   828.1  
C1                  1.8     1.9  
C2                  6.4     6.4  
C3                 18.4    17.8  
C4                 52.2    52.2  
C5                189.9   190.1  
C6                789.0   775.3  
C7               3309.3  3330.2

Графи з E >= 1,000,000: B7, B8, B9, B10, B11, G1, C4, C5, C6, C7
Відносний час (1.00 = найкраща комбінація блоків на цьому графі):


partition_block  4096                 8192                 16384              
filter_block      1024   2048   4096   1024   2048   4096   1024   2048   4096
graph                                                                         
B7               1.022  1.024  1.016  1.000  1.022  1.026  1.016  1.000  1.008
B8               1.007  1.019  1.013  1.008  1.001  1.005  1.000  1.005  1.003
B9               1.000  1.006  1.004  1.008  1.016  1.018  1.026  1.028  1.031
B10              1.007  1.005  1.002  1.001  1.001  1.000  1.031  1.032  1.037
B11              1.126  1.097  1.093  1.018  1.000  1.007  1.051  1.028  1.033
G1               1.101  1.007  1.002  1.000  1.002  1.031  1.089  1.018  1.017
C4               1.032  1.027  1.038  1.030  1.000  1.026  1.021  1.007  1.007
C5               1.026  1.023  1.033  1.001  1.021  1.000  1.028  1.013  1.014
C6               1.022  1.003  1.006  1.013  1.004  1.005  1.009  1.018  1.000
C7               1.011  1.007  1.005  1.013  1.006  1.005  1.008  1.000  1.006

середнє_відставання  найгірший_граф  перемог
partition_block filter_block                                              
8192            2048                        1.007           1.022        2
                1024                        1.009           1.030        1
                4096                        1.012           1.031        2
16384           2048                        1.015           1.032        2
                4096                        1.016           1.037        1
4096            4096                        1.021           1.093        0
                2048                        1.021           1.097        0
16384           1024                        1.028           1.089        1
4096            1024                        1.035           1.126        1


Найкращий розмір блоку: partition=8192, filter=2048
Він повільніший за найкращий на кожному графі в середньому на 0.7%
Типовий розкид між повторами: 1.4%  (різниця, менша за нього - це шум)
